# Predictions Test Analysis

Analysis of model predictions on the NIST23 test set, joining:
- `data/predicted/predictions_test.pkl` — model outputs and cosine similarity scores
- `data/proc/nist23/spec_df.pkl` — spectrum metadata
- `data/proc/nist23/mol_df.pkl` — molecule metadata
- `data/classyfire/nist23_mol_classyfire.pkl` — ClassyFire chemical taxonomy labels

In [1]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

DATA_ROOT = Path("../data")

## 1. Load Data

In [ ]:
with open(DATA_ROOT / "predicted/predictions_test.pkl", "rb") as f:
    pred_df = pickle.load(f)

with open(DATA_ROOT / "proc/nist23/spec_df.pkl", "rb") as f:
    spec_df = pickle.load(f)

with open(DATA_ROOT / "proc/nist23/mol_df.pkl", "rb") as f:
    mol_df = pickle.load(f)
mol_df = mol_df.drop(columns=["mol"], errors="ignore")

with open(DATA_ROOT / "classyfire/nist23_mol_classyfire.pkl", "rb") as f:
    cf_df = pickle.load(f)

print(f"pred_df:  {pred_df.shape}")
print(f"spec_df:  {spec_df.shape}")
print(f"mol_df:   {mol_df.shape}")
print(f"cf_df:    {cf_df.shape}")

In [ ]:
pred_df.head(3)

In [ ]:
spec_df.head(3)

In [ ]:
mol_df.head(3)

## 2. Merge DataFrames

In [ ]:
# Merge predictions with spectrum metadata
spec_meta_cols = [
    "spec_id", "mol_id", "prec_type", "inst_type", "frag_mode",
    "ion_mode", "dset", "ace", "nce", "ce_type",
]
df = pred_df.merge(spec_df[spec_meta_cols], on=["spec_id", "mol_id"], how="left")

# Merge with molecule metadata
mol_meta_cols = ["mol_id", "smiles", "formula", "mw", "exact_mw", "num_atoms", "num_bonds", "inchikey_s"]
df = df.merge(mol_df[mol_meta_cols], on="mol_id", how="left")

# Merge with ClassyFire labels
cf_meta_cols = ["mol_id", "superklass", "klass", "subklass", "direct_parent"]
df = df.merge(cf_df[cf_meta_cols], on="mol_id", how="left")

print(f"Merged df: {df.shape}")
df.head(3)

## 3. Overall Similarity Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, label in zip(
    axes,
    ["cos_sim_0.01", "cos_hun"],
    ["Cosine Sim (tol=0.01)", "Hungarian Cosine Sim"],
):
    ax.hist(df[col], bins=50, edgecolor="none", alpha=0.8)
    ax.axvline(df[col].mean(), color="red", linestyle="--", label=f"Mean={df[col].mean():.3f}")
    ax.axvline(df[col].median(), color="orange", linestyle="--", label=f"Median={df[col].median():.3f}")
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.set_title(f"Distribution of {label}")
    ax.legend()

plt.tight_layout()
plt.show()

print(df[["cos_sim_0.01", "cos_hun"]].describe().round(4))

## 4. Performance by Instrument Type

In [ ]:
inst_stats = (
    df.groupby("inst_type")[["cos_sim_0.01", "cos_hun"]]
    .agg(["mean", "median", "count"])
    .round(4)
)
inst_stats

In [ ]:
inst_order = df.groupby("inst_type")["cos_sim_0.01"].median().sort_values(ascending=False).index.tolist()
groups = [df.loc[df["inst_type"] == g, "cos_sim_0.01"].dropna().values for g in inst_order]

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot(groups, labels=inst_order, patch_artist=True, showfliers=False,
           boxprops=dict(facecolor="steelblue", alpha=0.6))
ax.set_title("Cosine Sim by Instrument Type")
ax.set_xlabel("Instrument Type")
ax.set_ylabel("Cosine Sim (tol=0.01)")
plt.tight_layout()
plt.show()

## 5. Performance by Precursor Type

In [ ]:
top_prec_types = df["prec_type"].value_counts().head(8).index
df_top = df[df["prec_type"].isin(top_prec_types)]

prec_stats = (
    df_top.groupby("prec_type")[["cos_sim_0.01", "cos_hun"]]
    .agg(["mean", "median", "count"])
    .round(4)
)
prec_stats

In [ ]:
prec_order = df_top.groupby("prec_type")["cos_sim_0.01"].median().sort_values(ascending=False).index.tolist()
groups = [df_top.loc[df_top["prec_type"] == g, "cos_sim_0.01"].dropna().values for g in prec_order]

fig, ax = plt.subplots(figsize=(10, 4))
ax.boxplot(groups, labels=prec_order, patch_artist=True, showfliers=False,
           boxprops=dict(facecolor="darkorange", alpha=0.6))
ax.set_title("Cosine Sim by Precursor Type (top 8)")
ax.set_xlabel("Precursor Type")
ax.set_ylabel("Cosine Sim (tol=0.01)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. Performance by Ion Mode

In [ ]:
ion_stats = (
    df[df["ion_mode"].isin(["P", "N"])]
    .groupby("ion_mode")[["cos_sim_0.01", "cos_hun"]]
    .agg(["mean", "median", "count"])
    .round(4)
)
ion_stats

## 7. Performance by Fragmentation Mode (HCD vs CID)

In [ ]:
frag_stats = (
    df.groupby("frag_mode")[["cos_sim_0.01", "cos_hun"]]
    .agg(["mean", "median", "count"])
    .round(4)
)
frag_stats

## 8. Effect of Collision Energy on Performance

In [ ]:
df_nce = df[(df["nce"].notna()) & (df["nce"] > 0) & (df["nce"] <= 200)].copy()
df_nce["nce_bin"] = pd.cut(df_nce["nce"], bins=[0, 20, 30, 40, 60, 80, 120, 200], right=True)

nce_stats = (
    df_nce.groupby("nce_bin", observed=True)[["cos_sim_0.01", "cos_hun"]]
    .agg(["mean", "median", "count"])
    .round(4)
)
nce_stats

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
nce_med = df_nce.groupby("nce_bin", observed=True)["cos_sim_0.01"].median()
nce_med.plot(kind="bar", ax=ax, color="steelblue", edgecolor="none")
ax.set_title("Median Cosine Sim by NCE Bin")
ax.set_xlabel("NCE Bin")
ax.set_ylabel("Median Cosine Sim (tol=0.01)")
ax.set_ylim(0, 1)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 9. Performance by Dataset Source

In [ ]:
dset_stats = (
    df.groupby("dset")[["cos_sim_0.01", "cos_hun"]]
    .agg(["mean", "median", "count"])
    .round(4)
)
dset_stats

## 10. Performance vs Molecular Properties

In [ ]:
df_mol = df[(df["mw"].notna()) & (df["num_atoms"].notna())].copy()
df_mol["mw_bin"] = pd.cut(df_mol["mw"], bins=[0, 150, 250, 350, 500, 750, 2000], right=True)
df_mol["natoms_bin"] = pd.cut(df_mol["num_atoms"], bins=[0, 10, 20, 30, 40, 60, 200], right=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

mw_med = df_mol.groupby("mw_bin", observed=True)["cos_sim_0.01"].median()
mw_med.plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="none")
axes[0].set_title("Median Cosine Sim by Molecular Weight")
axes[0].set_xlabel("MW Bin")
axes[0].set_ylabel("Median Cosine Sim")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=30)

na_med = df_mol.groupby("natoms_bin", observed=True)["cos_sim_0.01"].median()
na_med.plot(kind="bar", ax=axes[1], color="darkorange", edgecolor="none")
axes[1].set_title("Median Cosine Sim by Number of Atoms")
axes[1].set_xlabel("Num Atoms Bin")
axes[1].set_ylabel("Median Cosine Sim")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 11. Out-of-Spectrum Probability vs Similarity

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    df["pred_oos_prob"],
    df["cos_sim_0.01"],
    alpha=0.05,
    s=3,
    rasterized=True,
)
corr = df[["pred_oos_prob", "cos_sim_0.01"]].corr().iloc[0, 1]
ax.set_xlabel("Predicted OOS Probability")
ax.set_ylabel("Cosine Sim (tol=0.01)")
ax.set_title(f"OOS Prob vs Cosine Sim  (r={corr:.3f})")
plt.tight_layout()
plt.show()

In [ ]:
df["oos_bin"] = pd.cut(df["pred_oos_prob"], bins=[0, 0.05, 0.1, 0.2, 0.5, 1.0], right=True)
oos_stats = (
    df.groupby("oos_bin", observed=True)[["cos_sim_0.01", "cos_hun"]]
    .agg(["mean", "median", "count"])
    .round(4)
)
oos_stats

## 12. Top / Bottom Predictions

In [ ]:
display_cols = ["spec_id", "mol_id", "smiles", "formula", "prec_type", "inst_type", "cos_sim_0.01", "cos_hun"]

print("=== Top 10 predictions ===")
display(df.nlargest(10, "cos_sim_0.01")[display_cols].reset_index(drop=True))

print("=== Bottom 10 predictions ===")
display(df.nsmallest(10, "cos_sim_0.01")[display_cols].reset_index(drop=True))

## 13. Performance by ClassyFire Superclass

In [ ]:
cf_super_stats = (
    df.dropna(subset=["superklass"])
    .groupby("superklass")[["cos_sim_0.01", "cos_hun"]]
    .agg(["mean", "median", "count"])
    .round(4)
)
cf_super_stats.sort_values(("cos_sim_0.01", "median"), ascending=False)

In [ ]:
df_cf = df.dropna(subset=["superklass"])
super_order = df_cf.groupby("superklass")["cos_sim_0.01"].median().sort_values(ascending=False).index.tolist()
groups = [df_cf.loc[df_cf["superklass"] == g, "cos_sim_0.01"].values for g in super_order]
counts = [len(g) for g in groups]

fig, ax = plt.subplots(figsize=(14, 5))
ax.boxplot(groups, labels=[f"{s}\n(n={c:,})" for s, c in zip(super_order, counts)],
           patch_artist=True, showfliers=False,
           boxprops=dict(facecolor="steelblue", alpha=0.6))
ax.set_title("Cosine Sim by ClassyFire Superclass")
ax.set_ylabel("Cosine Sim (tol=0.01)")
ax.set_ylim(0, 1)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.tight_layout()
plt.show()

## 14. Performance by ClassyFire Class (Top 25)

In [ ]:
top25_classes = df["klass"].value_counts().head(25).index.tolist()
df_klass = df[df["klass"].isin(top25_classes)].dropna(subset=["klass"])

cf_klass_stats = (
    df_klass.groupby("klass")[["cos_sim_0.01", "cos_hun"]]
    .agg(["mean", "median", "count"])
    .round(4)
)
cf_klass_stats.sort_values(("cos_sim_0.01", "median"), ascending=False)

In [ ]:
klass_order = df_klass.groupby("klass")["cos_sim_0.01"].median().sort_values(ascending=False).index.tolist()
groups = [df_klass.loc[df_klass["klass"] == g, "cos_sim_0.01"].values for g in klass_order]
counts = [len(g) for g in groups]

fig, ax = plt.subplots(figsize=(18, 5))
ax.boxplot(groups, labels=[f"{k}\n(n={c:,})" for k, c in zip(klass_order, counts)],
           patch_artist=True, showfliers=False,
           boxprops=dict(facecolor="darkorange", alpha=0.6))
ax.set_title("Cosine Sim by ClassyFire Class (Top 25 by frequency)")
ax.set_ylabel("Cosine Sim (tol=0.01)")
ax.set_ylim(0, 1)
plt.xticks(rotation=55, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

## 15. Performance by ClassyFire Subclass (Top 25)

In [ ]:
top25_subklasses = df["subklass"].value_counts().head(25).index.tolist()
df_sub = df[df["subklass"].isin(top25_subklasses)].dropna(subset=["subklass"])

cf_sub_stats = (
    df_sub.groupby("subklass")[["cos_sim_0.01", "cos_hun"]]
    .agg(["mean", "median", "count"])
    .round(4)
)
cf_sub_stats.sort_values(("cos_sim_0.01", "median"), ascending=False)

In [ ]:
sub_order = df_sub.groupby("subklass")["cos_sim_0.01"].median().sort_values(ascending=False).index.tolist()
groups = [df_sub.loc[df_sub["subklass"] == g, "cos_sim_0.01"].values for g in sub_order]
counts = [len(g) for g in groups]

fig, ax = plt.subplots(figsize=(18, 5))
ax.boxplot(groups, labels=[f"{s}\n(n={c:,})" for s, c in zip(sub_order, counts)],
           patch_artist=True, showfliers=False,
           boxprops=dict(facecolor="mediumseagreen", alpha=0.6))
ax.set_title("Cosine Sim by ClassyFire Subclass (Top 25 by frequency)")
ax.set_ylabel("Cosine Sim (tol=0.01)")
ax.set_ylim(0, 1)
plt.xticks(rotation=55, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

## 16. Summary Table

In [ ]:
thresholds = [0.2, 0.5, 0.7, 0.9]

summary = {}
for t in thresholds:
    summary[f"cos_sim >= {t}"] = (df["cos_sim_0.01"] >= t).mean()

summary["mean cos_sim"] = df["cos_sim_0.01"].mean()
summary["median cos_sim"] = df["cos_sim_0.01"].median()
summary["mean cos_hun"] = df["cos_hun"].mean()
summary["median cos_hun"] = df["cos_hun"].median()
summary["n_spectra"] = len(df)
summary["n_molecules"] = df["mol_id"].nunique()

pd.Series(summary).round(4).to_frame("value")